In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
df = pd.read_csv("../data/processed/india_cost_quality_merged.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (116, 14)


,City,Average Rent (INR/month),Food Cost (INR/month),Internet Speed (Mbps),Healthcare Rating,Safety Score,Happiness Index,months_covered,cost_one_person_inr,rent_one_person_inr,monthly_salary_after_tax_inr,income_after_rent_inr,rent_difference_inr,rent_difference_percent
0,Mumbai,34896,6196,94.48,6.2,2.7,5.3,0.3,61943.98,35233.38,19505.76,-15727.62,337.38,0.966816
1,Delhi,26424,4487,130.26,6.1,2.6,5.6,0.5,42965.40,16694.12,20296.54,3602.42,-9729.88,36.822131
2,Bengaluru,26667,6936,81.36,8.7,6.7,7.5,0.6,40680.94,19417.90,27325.64,7907.74,-7249.10,27.183785
3,Hyderabad,25785,6375,108.84,7.8,7.7,6.3,0.6,41295.99,14936.85,25480.50,10543.66,-10848.15,42.071553
4,Ahmedabad,20881,3632,130.80,5.5,7.9,4.3,0.7,38835.80,13970.34,27237.78,13267.43,-6910.66,33.095446


We want K-Means to understand a city's:

cost
rent
food
digital connectivity
healthcare
safety
happiness

In [3]:
cluster_features = [
    "cost_one_person_inr",
    "Average Rent (INR/month)",
    "Food Cost (INR/month)",
    "Internet Speed (Mbps)",
    "Healthcare Rating",
    "Safety Score",
    "Happiness Index"
]

X_cluster = df[cluster_features]

X_cluster.head()

,cost_one_person_inr,Average Rent (INR/month),Food Cost (INR/month),Internet Speed (Mbps),Healthcare Rating,Safety Score,Happiness Index
0,61943.98,34896,6196,94.48,6.2,2.7,5.3
1,42965.40,26424,4487,130.26,6.1,2.6,5.6
2,40680.94,26667,6936,81.36,8.7,6.7,7.5
3,41295.99,25785,6375,108.84,7.8,7.7,6.3
4,38835.80,20881,3632,130.80,5.5,7.9,4.3


In [4]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_cluster)

X_scaled[:5]

array([[ 3.83049319,  4.66534933,  0.49123804,  0.25685977, -0.15704332,
        -1.32201089, -0.79543742],
       [ 1.43131241,  3.16890242, -0.72048531,  1.31973062, -0.2271088 ,
        -1.37240631, -0.56571773],
       [ 1.14252197,  3.21182459,  1.01591637, -0.13287934,  1.5945937 ,
         0.69380591,  0.88917361],
       [ 1.22027363,  3.05603302,  0.61815347,  0.68343399,  0.96400437,
         1.19776011, -0.02970513],
       [ 0.90926824,  2.18981776, -1.32670149,  1.33577171, -0.64750168,
         1.29855095, -1.5611697 ]])

In [5]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

clusters[:10]

array([3, 3, 3, 3, 3, 3, 3, 3, 1, 2], dtype=int32)

In [6]:
df["city_cluster"] = clusters

df[["City", "city_cluster"]].head(10)

,City,city_cluster
0,Mumbai,3
1,Delhi,3
2,Bengaluru,3
3,Hyderabad,3
4,Ahmedabad,3
5,Chennai,3
6,Kolkata,3
7,Pune,3
8,Jaipur,1
9,Surat,2


In [7]:
cluster_summary = df.groupby("city_cluster")[
    [
        "Average Rent (INR/month)",
        "Food Cost (INR/month)",
        "Internet Speed (Mbps)",
        "Healthcare Rating",
        "Safety Score",
        "Happiness Index"
    ]
].mean()

cluster_summary

,Average Rent (INR/month),Food Cost (INR/month),Internet Speed (Mbps),Healthcare Rating,Safety Score,Happiness Index
city_cluster,,,,,,
0,7990.806452,6151.516129,82.646129,6.196774,6.570968,4.880645
1,6390.217391,4276.652174,82.247391,5.004348,5.560870,6.456522
2,6603.040816,5592.612245,82.231633,7.151020,4.342857,7.204082
3,20450.153846,5789.923077,113.352308,6.738462,5.623077,6.346154


What the model appears to have discovered

Cluster 0 — Higher-cost, safety-oriented

Relatively higher rent/food among the lower-rent clusters
Highest safety among these groups
Lower happiness

Cluster 1 — Low-cost / relatively balanced

Lowest rent
Lowest food cost
Moderate quality scores
Good happiness

Cluster 2 — Quality/livability-oriented

Low rent
Good food cost
Highest healthcare
Highest happiness
But lowest safety

Cluster 3 — High-rent / digitally stronger cities

Rent is dramatically higher
Highest internet speed
Moderate-to-good healthcare, safety and happiness

Learn the natural city segments from cost + quality-of-life characteristics.

                 YOUR 116 CITIES
                       │
                       ▼
              6 CITY FEATURES
                       │
                       ▼
                 StandardScaler
                       │
                       ▼
                    X_scaled
                       │
             ┌─────────┴─────────┐
             │                   │
           K = 2               K = 3
             │                   │
         K-Means              K-Means
             │                   │
       inertia +             inertia +
       silhouette             silhouette
             │                   │
             └─────────┬─────────┘
                       │
                     ...
                       │
                    K = 8
                       │
                       ▼
             Compare the results
                       │
                       ▼
              Choose sensible K
                       │
                       ▼
          FINAL K-MEANS MODEL
                       │
                       ▼
             City Clusters/Profile

In [8]:
from sklearn.metrics import silhouette_score

inertias = []
silhouette_scores = []
k_values = range(2, 9)

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = model.fit_predict(X_scaled)
    
    inertias.append(model.inertia_)
    silhouette_scores.append(
        silhouette_score(X_scaled, labels)
    )

for k, inertia, score in zip(k_values, inertias, silhouette_scores):
    print(f"k={k}: inertia={inertia:.2f}, silhouette={score:.3f}")

k=2: inertia=683.27, silhouette=0.298
k=3: inertia=589.29, silhouette=0.149
k=4: inertia=534.92, silhouette=0.146
k=5: inertia=486.22, silhouette=0.146
k=6: inertia=445.75, silhouette=0.154
k=7: inertia=417.67, silhouette=0.149
k=8: inertia=386.52, silhouette=0.155


In [10]:
from sklearn.cluster import KMeans

# Final K-Means model
kmeans = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
)

# Train K-Means and assign each city to a cluster
clusters = kmeans.fit_predict(X_scaled)

# Add cluster labels to our dataset
df["city_cluster"] = clusters

# Check assignments
df[["City", "city_cluster"]].head(10)

,City,city_cluster
0,Mumbai,0
1,Delhi,0
2,Bengaluru,0
3,Hyderabad,0
4,Ahmedabad,0
5,Chennai,0
6,Kolkata,0
7,Pune,0
8,Jaipur,1
9,Surat,1


In [11]:
cluster_summary = df.groupby("city_cluster")[
    [
        "Average Rent (INR/month)",
        "Food Cost (INR/month)",
        "Internet Speed (Mbps)",
        "Healthcare Rating",
        "Safety Score",
        "Happiness Index",
        "months_covered"
    ]
].mean()

cluster_summary

,Average Rent (INR/month),Food Cost (INR/month),Internet Speed (Mbps),Healthcare Rating,Safety Score,Happiness Index,months_covered
city_cluster,,,,,,,
0,19541.000000,5664.266667,114.306000,6.900000,5.920000,6.606667,0.800000
1,6841.346535,5479.237624,81.604554,6.353465,5.234653,6.299010,1.145545


In [12]:
df[["City", "city_cluster"]].sort_values(
    "city_cluster"
).to_string(index=False)

'              City  city_cluster\n            Mumbai             0\n             Delhi             0\n         Bengaluru             0\n         Hyderabad             0\n         Ahmedabad             0\n           Chennai             0\n           Kolkata             0\n              Pune             0\n             Patna             0\nThiruvananthapuram             0\n        Chandigarh             0\n           Madurai             0\n       Tirunelveli             0\n             Alwar             0\n             Kochi             0\n            Nagpur             1\n            Bhopal             1\n     Visakhapatnam             1\n          Vadodara             1\n         Ghaziabad             1\n          Ludhiana             1\n              Agra             1\n            Nashik             1\n         Faridabad             1\n            Meerut             1\n            Rajkot             1\n          Varanasi             1\n          Srinagar             1\n        Auran

Cluster 0 → High-cost / higher-service urban profile

Cluster 1 → Lower-cost / lower-service urban profile                                  K-Means has created:

Cluster 0: only 16 cities
Cluster 1: the remaining 100 cities

"Silhouette analysis indicated k=2 as the strongest separation, while k=4 was selected for the SmartShift system because it provides more granular and actionable city profiles for personalized recommendations."

# Supervised ML: Cost of Living Estimation

target = "cost_one_person_inr"

features = [
    "Average Rent (INR/month)",
    "Food Cost (INR/month)",
    "Internet Speed (Mbps)",
    "Healthcare Rating",
    "Safety Score",
    "Happiness Index"
]

X = df[features]
y = df[target]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

print("\nX shape:", X.shape)
print("y shape:", y.shape)